# CER du 1B fine-tuné — mesuré comme au benchmark 0003 (Kaggle ou RunPod)

Issue #474 · critère d'acceptation de BAO-2 (`docs/wouri-planning/10`) · suite de la fiche `docs/benchmarks/0006`.

**But** : mesurer le **CER** du modèle fine-tuné retenu (`step_1200`) sur **les mêmes 290 clips**, avec **la même
normalisation** et **le même pipeline** que le benchmark 0003, pour le comparer directement aux **22,1 %** du 1B de départ.

**Témoin** : le 1B de départ est remesuré dans la même session. Il doit retrouver ~22,1 % (± 0,5 point) ; sinon
l'environnement diffère du benchmark 0003 et la comparaison n'est pas recevable (on le publie tel quel).

**Avant de lancer** — il faut un jeton Hugging Face **en lecture seule** (le checkpoint est dans un dépôt privé) :
- **Kaggle** (gratuit) : *Settings → Accelerator → **GPU T4 x2*** (un seul est utilisé), *Settings → **Internet on***, et
  *Add-ons → **Secrets*** : un secret nommé **`HF_TOKEN`** contenant le jeton.
- **RunPod** (payant, ~0,30 $/h) : un GPU d'au moins 16 Go (ex. RTX A5000), template PyTorch avec Jupyter. Le jeton est
  demandé par une **saisie masquée** à la cellule 6.

Le jeton n'est jamais écrit dans le notebook ni dans les journaux.

Le Python du noyau n'a pas d'importance : le notebook crée son **propre Python 3.12** (fairseq2 0.6 n'existe pas en 3.13).
**Exécuter les cellules dans l'ordre.** Si le noyau redémarre : ré-exécuter la cellule 1, puis reprendre.

In [ ]:
# Cellule 1 — chemins, réglages et lanceur commun (re-exécuter si le noyau redémarre)
import os, shutil, subprocess, sys
from pathlib import Path

# RunPod : tout sur le volume /workspace. Kaggle : /kaggle/working est limité à 20 Go -> gros fichiers dans /tmp.
RUNPOD = Path("/workspace").is_dir()
WORK = Path("/workspace") if RUNPOD else Path.cwd()   # /kaggle/working sur Kaggle
os.chdir(WORK)
BIG = WORK if RUNPOD else Path("/tmp")
VENV = BIG / "omni312"                    # Python 3.12 dédié (fairseq2 0.6 n'existe pas en 3.13)
VENV_PY = str(VENV / "bin" / "python")
CKPT_ROOT = BIG / "wourri_ft"             # checkpoint téléchargé (~3,9 Go)
HF_REPO = "seven990/omniasr-ctc-1b-bci-wourri"   # dépôt Hugging Face PRIVÉ du fine-tune (fiche 0006)
STEP = "step_1200"                        # version retenue sur dev (fiche 0006)
RESULTS = WORK / "eval_bci_cer_results.json"
# Téléchargement Hugging Face normal (hf_transfer n'est pas installé dans le Python 3.12)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

def clean_env(extra=None):
    """Environnement des sous-processus SANS jetons (fairseq2 peut journaliser les variables)."""
    env = {k: v for k, v in os.environ.items()
           if not any(s in k.upper() for s in ("TOKEN", "SECRET", "PASSWORD", "CREDENTIAL", "API_KEY"))}
    env["PYTHONUNBUFFERED"] = "1"
    env.update(extra or {})
    return env

def sh(args, cwd=None, env=None, log_path=None, check=True):
    """Lance une commande en affichant sa sortie en direct (copiée dans log_path) ; retourne le code."""
    proc = subprocess.Popen([str(a) for a in args], cwd=cwd, env=env or clean_env(), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    log = open(log_path, "w", encoding="utf-8") if log_path else None
    for line in proc.stdout:
        print(line, end="")
        if log:
            log.write(line)
    if log:
        log.close()
    code = proc.wait()
    if check and code != 0:
        raise SystemExit(f"ECHEC (code {code}) : {' '.join(str(a) for a in args)[:200]}")
    return code

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout
      or "PAS DE GPU -> Kaggle : Settings > Accelerator > GPU T4 x2 | RunPod : choisir un pod GPU")
print("Python du noyau :", sys.version.split()[0], "(sans importance : la mesure tourne dans", VENV, ")")
print(">>> CELLULE 1 OK -> cellule 2.")

## Étape 1 — Installation (Python 3.12 dédié, ~5 à 7 min)

Mêmes versions que le benchmark 0003 : `omnilingual-asr==0.1.0`, `fairseq2[arrow]==0.6`, `torch==2.8.0`.
`pyarrow==20.0.0` : un pyarrow plus récent exige NumPy 2 (constaté le 2026-10-19).

In [ ]:
# Cellule 2 — Python 3.12 (uv) + versions figées du benchmark 0003 + jiwer (score)
# --no-cache : évite une 2e copie de torch + CUDA (~6 Go) sur le disque de la session.
UV = [sys.executable, "-m", "uv"]
sh([sys.executable, "-m", "pip", "install", "--quiet", "uv"])
if not VENV.exists():
    sh(UV + ["venv", "--quiet", "--python", "3.12", VENV])
sh(UV + ["pip", "install", "--quiet", "--no-cache", "--python", VENV_PY,
         "omnilingual-asr==0.1.0", "fairseq2[arrow]==0.6", "torch==2.8.0", "torchaudio==2.8.0",
         "datasets<4.0", "soundfile", "librosa<1.0", "jiwer", "pyarrow==20.0.0"])
sh([VENV_PY, "-c", "import sys, torch, jiwer, omnilingual_asr; print('Python', sys.version.split()[0], "
    "'| torch', torch.__version__, '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN')"])
print("\n>>> INSTALLATION TERMINÉE (pas de redémarrage) -> cellule 3.")

## Étape 2 — Les modules du dépôt

Les cellules 3, 4 et 5 écrivent trois fichiers du dépôt (`finetune/`), copies identiques vérifiées par un test :
la révision du dataset (`prepare_bci_dataset.py`, qui s'appuie sur `dictee_to_parquet.py`) et la mesure
(`eval_bci_cer.py` : normalisation et score copiés du benchmark 0003). Il suffit de les exécuter.

In [ ]:
%%writefile dictee_to_parquet.py
"""Convertit l'export dictée (ZIP audiofolder) en dataset parquet Omnilingual — maillon ③ (#474).

Pont entre le maillon ② (export dictée, ADR-0035) et le maillon ③ (fine-tune
Omnilingual, cf. étude de faisabilité docs/benchmarks/0004 et test mécanique 0005).

Entrée : ZIP export dictée
    audio/<x>.webm ...
    metadata.csv : file_name, transcription, language, filiere, text_fr

Sortie : un dataset parquet partitionné (« hive »), seul format que lit le chargeur
`MixtureParquetStorage` de la recette de fine-tune (omnilingual-asr 0.1.0) :
    <racine>/corpus=<corpus>/split=<split>/language=<code>/part-0.parquet
        - text        : transcription normalisée (baoulé) — la CIBLE ASR
        - audio_bytes : audio FLAC 16 kHz mono, en list<int8> (schéma amont)
        - audio_size  : nombre d'échantillons du waveform décodé
    corpus / split / language vivent dans les NOMS DE DOSSIERS, pas dans le fichier :
    la recette découvre les splits et les partitions à partir de l'arborescence.
    Un fichier parquet « à plat » n'est pas lu (aucun split découvert).

Le code langue est celui d'Omnilingual (ISO 639-3 + écriture) : `bci` -> `bci_Latn`.

Usage :
    python dictee_to_parquet.py export.zip dataset_root \\
        --corpus wourri_dictee --split train --language bci \\
        --stats language_distribution_0.tsv

Décodage : soundfile (wav/flac/ogg) ; repli librosa+ffmpeg pour webm/opus/mp3
(le fine-tune tourne sur Colab/Kaggle où ffmpeg est présent). Le rééchantillonnage
utilise librosa si présent, sinon un repli linéaire (dégradé — le vrai run a librosa).

Ce module est recopié tel quel dans le notebook `colab/omnilingual_finetune_smoke_test.ipynb`
(cellule %%writefile) : un test vérifie que les deux copies restent identiques.
"""
from __future__ import annotations

import argparse
import csv
import io
import re
import sys
import unicodedata
import zipfile
from pathlib import Path, PurePosixPath

TARGET_SR = 16000
ROW_GROUP_SIZE = 100  # valeur amont : limite la mémoire du streaming et permet le mélange
FILE_COLUMNS = ["text", "audio_bytes", "audio_size"]

# Apostrophe baoulé : Common Voice bci et la sortie d'Omnilingual l'écrivent U+02BC (ʼ,
# lettre modificative, conservée par la normalisation amont et présente dans le vocabulaire :
# le 1B la produit au benchmark 0003). Les variantes tapées au clavier ENTRE DEUX LETTRES
# (' ’ ‘ ‛) y sont ramenées, sinon un même son aurait plusieurs cibles ; ailleurs ce sont
# des guillemets, retirés comme la ponctuation. Un ton combinant (ɛ̀ = ɛ + U+0300, sans forme
# précomposée) compte comme lettre.
_APOSTROPHES = "'‘’‛"
_INNER_APOSTROPHE = re.compile(rf"(?<=[\ẁ-ͯ])[{_APOSTROPHES}](?=\w)")
_OTHER_APOSTROPHE = re.compile(rf"[{_APOSTROPHES}]")
# Invisibles supprimés par l'amont (shared_deletion_list) + BOM : souvent hérités d'un copier-coller.
_INVISIBLE = re.compile("[​‌‎‏‪‬﻿]")
_PUNCT = re.compile(r"[.,!?;:«»\"“”„()\[\]{}…—–/\\_]")
_DIGIT_WORD = re.compile(r"(?<!\S)\d+(?!\S)")


def normalize_text(text: str) -> str:
    """Normalise une transcription comme la préparation de données amont (text_tools) :
    NFKC, invisibles supprimés, minuscules, ponctuation retirée, mots uniquement numériques
    retirés. Les lettres baoulé (ɛ ɔ ɲ, tons, ʼ) sont conservées : elles portent le sens."""
    t = unicodedata.normalize("NFKC", text or "")
    t = _INVISIBLE.sub("", t).lower()
    t = _INNER_APOSTROPHE.sub("ʼ", t)
    t = _OTHER_APOSTROPHE.sub(" ", t)
    t = _PUNCT.sub(" ", t)
    t = _DIGIT_WORD.sub(" ", t)
    return re.sub(r"\s+", " ", t).strip()


def omni_language_code(code: str) -> str:
    """Code Omnilingual (`lang_ids.py`) = ISO 639-3 + écriture : 'bci' -> 'bci_Latn'.

    Les langues de l'atelier s'écrivent en latin (colonne `script` de la table
    `languages`) ; un code déjà complet ('bci_Latn') est gardé tel quel."""
    code = (code or "").strip()
    if not code:
        raise ValueError("code langue vide")
    return code if "_" in code else f"{code}_Latn"


def decode_to_16k_mono(raw: bytes, filename: str):
    """Octets audio -> (waveform float32 mono 16 kHz, n_samples)."""
    import numpy as np
    import soundfile as sf

    try:
        wav, sr = sf.read(io.BytesIO(raw), dtype="float32", always_2d=True)  # (n, ch)
        wav = wav.mean(axis=1)  # -> mono (n,)
    except Exception:
        wav, sr = _decode_via_librosa(raw, filename)  # webm/opus/mp3 -> ffmpeg
    if sr != TARGET_SR:
        wav = _resample(np.asarray(wav, dtype="float32"), sr, TARGET_SR)
    return np.asarray(wav, dtype="float32"), int(len(wav))


def _decode_via_librosa(raw: bytes, filename: str):
    import tempfile
    import warnings

    import librosa

    suffix = Path(filename).suffix or ".webm"
    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
        tmp.write(raw)
        path = tmp.name
    try:
        # webm : soundfile échoue toujours, librosa passe par audioread + ffmpeg. Ses deux
        # avertissements (« PySoundFile failed », dépréciation d'audioread) sortiraient une fois
        # par clip et noieraient la sortie du notebook (433 clips = 433 paires).
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            warnings.simplefilter("ignore", FutureWarning)
            wav, sr = librosa.load(path, sr=None, mono=True)  # float32 mono
    finally:
        Path(path).unlink(missing_ok=True)
    return wav, sr


def _resample(wav, sr: int, target: int):
    import numpy as np

    try:
        import librosa

        return librosa.resample(wav, orig_sr=sr, target_sr=target)
    except Exception:
        # repli linéaire (le vrai run a librosa ; ce repli garde le script utilisable en test)
        n = int(round(len(wav) * target / sr))
        if n <= 0:
            return wav
        return np.interp(
            np.linspace(0, len(wav) - 1, n), np.arange(len(wav)), wav
        ).astype("float32")


def encode_flac(wav) -> bytes:
    """Waveform float32 mono 16 kHz -> octets FLAC."""
    import soundfile as sf

    buf = io.BytesIO()
    sf.write(buf, wav, TARGET_SR, format="FLAC")
    return buf.getvalue()


def read_export(zip_path) -> list[dict]:
    """Lit le ZIP export dictée -> [{file_name, transcription, language, filiere, text_fr, raw}]."""
    rows: list[dict] = []
    with zipfile.ZipFile(zip_path) as zf:
        names = set(zf.namelist())
        metas = sorted(n for n in names if PurePosixPath(n).name == "metadata.csv")
        if not metas:
            raise SystemExit("metadata.csv introuvable dans le ZIP")
        meta_name = "metadata.csv" if "metadata.csv" in names else metas[0]
        meta = zf.read(meta_name).decode("utf-8-sig")
        for r in csv.DictReader(io.StringIO(meta)):
            fn = (r.get("file_name") or "").strip()
            if not fn:
                continue
            if fn not in names:
                # tolère un préfixe de chemin différent, jamais un autre fichier :
                # même nom exact (pas « 11.webm » pour « 1.webm ») et un seul candidat
                base = PurePosixPath(fn).name
                cands = [n for n in names if PurePosixPath(n).name == base]
                if len(cands) != 1:
                    why = "absent du ZIP" if not cands else f"ambigu ({len(cands)} fichiers)"
                    print(f"  [SKIP] audio {why} : {fn}", file=sys.stderr)
                    continue
                fn = cands[0]
            rows.append(
                {
                    "file_name": fn,
                    "transcription": (r.get("transcription") or "").strip(),
                    "language": (r.get("language") or "").strip(),
                    "filiere": (r.get("filiere") or "").strip(),
                    "text_fr": (r.get("text_fr") or "").strip(),
                    "raw": zf.read(fn),
                }
            )
    return rows


def make_table(examples):
    """[(transcription, waveform float32 mono 16 kHz)] -> table parquet (colonnes FILE_COLUMNS).

    Ignore les paires sans texte après normalisation ou sans audio : une paire
    audio↔texte complète est requise pour l'apprentissage."""
    import numpy as np
    import pyarrow as pa

    texts, audios, sizes = [], [], []
    for text, wav in examples:
        norm = normalize_text(text)
        if not norm or len(wav) == 0:
            continue
        texts.append(norm)
        audios.append(np.frombuffer(encode_flac(wav), dtype=np.int8))
        sizes.append(int(len(wav)))
    return pa.table(
        {
            "text": pa.array(texts, type=pa.string()),
            "audio_bytes": pa.array(audios, type=pa.list_(pa.int8())),
            "audio_size": pa.array(sizes, type=pa.int64()),
        }
    )


def write_partition(table, root, corpus: str, split: str, language: str) -> Path:
    """Écrit la table dans <root>/corpus=…/split=…/language=…/part-0.parquet.

    Remplace les parquet déjà présents dans CETTE partition : un export dictée est
    toujours complet, le reconvertir doit remplacer l'ancien, pas s'y ajouter."""
    import pyarrow.parquet as pq

    for key, value in (("corpus", corpus), ("split", split), ("language", language)):
        if not value or "/" in value or "=" in value:
            raise ValueError(f"{key} invalide pour un nom de partition : {value!r}")
    if "_" in split:
        # la recette lit un split « <split>_<corpus> » comme un filtre de corpus
        raise ValueError(f"split sans '_' attendu (reçu {split!r})")
    if table.num_rows == 0:
        # une partition vide fait échouer la recette au démarrage (lecture de son 1er exemple)
        raise ValueError(f"aucune paire audio↔texte exploitable pour {corpus}/{split}")
    part_dir = Path(root) / f"corpus={corpus}" / f"split={split}" / f"language={language}"
    part_dir.mkdir(parents=True, exist_ok=True)
    for old in part_dir.glob("*.parquet"):
        old.unlink()
    out = part_dir / "part-0.parquet"
    pq.write_table(table, out, row_group_size=ROW_GROUP_SIZE)
    return out


def write_stats(root, out_tsv) -> list[dict]:
    """Écrit le TSV `corpus / language / hours` attendu par `dataset_summary_path`
    (pondération des partitions à l'entraînement). Comme l'outil amont, les heures
    sont cumulées sur tous les splits."""
    import pyarrow.dataset as ds

    table = ds.dataset(str(root), format="parquet", partitioning="hive").to_table(
        columns=["corpus", "language", "audio_size"]
    )
    samples: dict[tuple[str, str], int] = {}
    for corpus, language, size in zip(
        table.column("corpus").to_pylist(),
        table.column("language").to_pylist(),
        table.column("audio_size").to_pylist(),
    ):
        key = (str(corpus), str(language))
        samples[key] = samples.get(key, 0) + int(size)
    stats = [
        {"corpus": c, "language": lang, "hours": n / TARGET_SR / 3600}
        for (c, lang), n in sorted(samples.items())
    ]
    lines = ["corpus\tlanguage\thours"]
    lines += [f"{s['corpus']}\t{s['language']}\t{s['hours']:.6f}" for s in stats]
    Path(out_tsv).parent.mkdir(parents=True, exist_ok=True)
    Path(out_tsv).write_text("\n".join(lines) + "\n", encoding="utf-8")
    return stats


def build_dataset(rows, root, corpus: str, split: str, language: str) -> int:
    """Lignes d'export dictée -> partition parquet. Retourne le nombre de clips écrits."""

    def examples():
        for r in rows:
            if r.get("transcription"):
                wav, _ = decode_to_16k_mono(r["raw"], r["file_name"])
                yield r["transcription"], wav

    table = make_table(examples())
    write_partition(table, root, corpus, split, omni_language_code(language))
    return table.num_rows


def main():
    ap = argparse.ArgumentParser(
        description="Export dictée (ZIP audiofolder) -> dataset parquet Omnilingual (#474)"
    )
    ap.add_argument("zip_path", help="ZIP export dictée (audio/ + metadata.csv)")
    ap.add_argument("out_root", help="racine du dataset (le `data:` de la carte fairseq2)")
    ap.add_argument("--corpus", default="wourri_dictee")
    ap.add_argument("--split", default="train", choices=["train", "dev", "test"])
    ap.add_argument("--language", default=None, help="code langue (ex. bci) ; défaut : celui du CSV")
    ap.add_argument("--stats", default=None, help="écrit aussi le TSV corpus/language/hours")
    args = ap.parse_args()

    rows = read_export(args.zip_path)
    language = args.language
    if not language:
        found = {r["language"] for r in rows if r.get("language")}
        if len(found) != 1:
            raise SystemExit(f"--language requis : langues trouvées dans le CSV = {sorted(found)}")
        language = found.pop()
    n = build_dataset(rows, args.out_root, args.corpus, args.split, language)
    print(f"OK : {n} clips -> {args.out_root} (corpus={args.corpus}, split={args.split}, "
          f"language={omni_language_code(language)})")
    if args.stats:
        write_stats(args.out_root, args.stats)
        print(f"Stats -> {args.stats}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile prepare_bci_dataset.py
"""Prépare le dataset baoulé du fine-tune Omnilingual (#474, ADR-0044).

Common Voice bci (miroir Klayt, révision épinglée du benchmark 0003) :
    train -> split « train », validation -> « dev », test -> « test » (avec --with-test)
+ l'export de l'écran Dictée de l'atelier (optionnel) -> corpus « wourri_dictee », split « train »,
puis le TSV de statistiques et la carte dataset fairseq2.

Le split « test » (290 clips, ceux du benchmark 0003) n'est JAMAIS entraîné : la recette
n'entraîne que sur `train_split: train`. Il est seulement évalué (`valid_split: dev,test`) pour
comparer, dans les mêmes conditions, le modèle de départ et le modèle fine-tuné. La séparation
locuteurs / phrases entre ces splits est vérifiée (docs/benchmarks/0005 §3).

Usage (dans le Python 3.12 du notebook) :
    python prepare_bci_dataset.py --work /workspace --dictee /workspace --with-test

Ce module est recopié tel quel dans les notebooks de finetune/colab/ (cellule %%writefile) :
`python finetune/colab/sync_smoke_test_notebook.py` les resynchronise, un test le vérifie.
"""
from __future__ import annotations

import argparse
import shutil
from pathlib import Path

import dictee_to_parquet as d2p

DATASET_REPO = "Klayt/baoule-common-voice"
REV = "176d5f8ad6c04e5ca1a0e66770866709bcbb338b"  # même révision que le benchmark 0003
# split Hugging Face -> split de la recette (« dev » : nom attendu par valid_split)
CV_SPLITS = {"train": "train", "validation": "dev", "test": "test"}
CARD_NAME = "wourri_bci"


def get_audio(row):
    """Audio d'une ligne Hugging Face -> (waveform float32 mono, sample rate)."""
    import numpy as np

    a = row["audio"]
    if isinstance(a, dict):  # datasets < 4.0
        return np.asarray(a["array"], dtype="float32"), int(a["sampling_rate"])
    samples = a.get_all_samples()  # datasets >= 4.0 (AudioDecoder)
    return samples.data.numpy().squeeze().astype("float32"), int(samples.sample_rate)


def cv_examples(hf_split, load_dataset=None):
    """[(phrase, waveform 16 kHz)] du split Common Voice demandé."""
    if load_dataset is None:
        from datasets import load_dataset
    for row in load_dataset(DATASET_REPO, revision=REV, split=hf_split):
        wav, sr = get_audio(row)
        if sr != d2p.TARGET_SR:
            wav = d2p._resample(wav, sr, d2p.TARGET_SR)
        yield row["sentence"], wav


def find_export(path, tmp_dir) -> Path:
    """Le ZIP de l'écran Dictée, ou son dossier décompressé (rezippé au format attendu)."""
    src = Path(path)
    if src.is_file():
        return src
    zips = sorted(src.rglob("dictee_*.zip"))
    if zips:
        return zips[0]
    metas = sorted(src.rglob("metadata.csv"))
    if metas:
        base = Path(tmp_dir) / "dictee_export"
        return Path(shutil.make_archive(str(base), "zip", root_dir=metas[0].parent))
    raise SystemExit(f"Ni dictee_*.zip ni metadata.csv sous {path}")


def write_card(asset_dir, data_root) -> Path:
    """Carte fairseq2 `mixture_parquet_asr_dataset` (répertoire FAIRSEQ2_USER_ASSET_DIR)."""
    asset_dir = Path(asset_dir)
    asset_dir.mkdir(parents=True, exist_ok=True)
    card = asset_dir / f"{CARD_NAME}.yaml"
    card.write_text(
        f"name: {CARD_NAME}\n"
        "dataset_family: mixture_parquet_asr_dataset\n"
        "dataset_config:\n"
        f"  data: {data_root}\n"
        "tokenizer_ref: omniASR_tokenizer\n",
        encoding="utf-8",
    )
    return card


def prepare(work, dictee=None, with_test=False, load_dataset=None) -> dict:
    """Construit <work>/wourri_bci (dataset partitionné + TSV) et la carte fairseq2.

    Repart toujours d'un dossier vide : une préparation précédente (autres splits,
    ancien export) ne doit pas se mélanger à celle-ci. Retourne les comptes par partition.
    """
    work = Path(work)
    dataset_dir = work / "wourri_bci"
    data_root = dataset_dir / "version=0"  # le `data:` de la carte fairseq2
    shutil.rmtree(dataset_dir, ignore_errors=True)
    lang = d2p.omni_language_code("bci")
    counts = {}

    for hf_split, split in CV_SPLITS.items():
        if split == "test" and not with_test:
            continue
        table = d2p.make_table(cv_examples(hf_split, load_dataset))
        d2p.write_partition(table, data_root, "cv_bci", split, lang)
        counts[f"cv_bci/{split}"] = table.num_rows

    if dictee:
        export = find_export(dictee, work)
        rows = d2p.read_export(export)
        counts["wourri_dictee/train"] = d2p.build_dataset(rows, data_root, "wourri_dictee", "train", "bci")

    stats = d2p.write_stats(data_root, dataset_dir / "language_distribution_0.tsv")
    card = write_card(work / "fairseq2_assets", data_root)
    return {"counts": counts, "stats": stats, "card": card}


def main():
    ap = argparse.ArgumentParser(description="Dataset baoulé du fine-tune Omnilingual (#474)")
    ap.add_argument("--work", required=True, help="dossier de travail (sortie)")
    ap.add_argument("--dictee", default=None, help="ZIP de la dictée ou dossier qui le contient")
    ap.add_argument("--with-test", action="store_true",
                    help="ajoute le split test (évalué, jamais entraîné)")
    args = ap.parse_args()
    result = prepare(args.work, args.dictee, args.with_test)
    for name, n in result["counts"].items():
        print(f"{name}: {n} clips")
    for s in result["stats"]:
        print(f"{s['corpus']:15s} {s['language']}  {s['hours']:.2f} h")
    print("Carte :", result["card"])


if __name__ == "__main__":
    main()


In [ ]:
%%writefile eval_bci_cer.py
"""CER/WER baoulé d'un modèle Omnilingual, mesuré À L'IDENTIQUE du benchmark 0003 (#474, BAO-2).

Critère de BAO-2 (docs/wouri-planning/10) : le CER du modèle fine-tuné, mesuré sur le même jeu
(Common Voice bci test, 290 clips) et comparé aux 22,1 % du 1B de départ. L'UER de la recette
d'entraînement (fiche 0006) n'utilise pas la même normalisation : ce module refait la mesure avec
le protocole du benchmark 0003 :
- mêmes clips : révision épinglée, split test complet, aucun clip exclu ;
- même normalisation et même score (`normalize`, `score` : copie exacte de la cellule 3 de
  finetune/colab/asr_baoule_benchmark.ipynb, vérifiée par un test) ;
- même pipeline : ASRInferencePipeline, un clip à la fois, langue bci_Latn, dtype par défaut.

Le 1B de départ est remesuré dans la même session (témoin) : s'il ne retrouve pas ~22,1 %,
l'environnement diffère de celui du benchmark et la comparaison n'est pas recevable.

Le checkpoint fine-tuné est chargé par une carte fairseq2 qui hérite d'omniASR_CTC_1B (famille,
architecture, tokenizer) et pointe vers le dossier `step_N/model` écrit par l'entraînement
(pp_00/tp_00/sdp_00.pt, lu par le chargeur natif de fairseq2 0.6).

Usage (dans le Python 3.12 du notebook, après téléchargement du checkpoint) :
    python eval_bci_cer.py --checkpoint-root /tmp/wourri_ft --step step_1200

Ce module est recopié tel quel dans le notebook d'évaluation (cellule %%writefile) :
`python finetune/colab/sync_smoke_test_notebook.py` le resynchronise, un test le vérifie.
"""
from __future__ import annotations

import argparse
import gc
import json
import os
import re
import tempfile
import time
import unicodedata
from pathlib import Path

import jiwer

from prepare_bci_dataset import DATASET_REPO, REV, get_audio

BASE_CARD = "omniASR_CTC_1B"
# CER du 1B de départ au benchmark 0003 (docs/benchmarks/0003-asr-baoule-evaluation-results.md §1)
BENCHMARK_0003_CER = 0.221
BENCHMARK_0003_N = 290
# Écart toléré pour le témoin : au-delà, l'environnement n'est pas celui du benchmark 0003
CONTROL_TOLERANCE = 0.005
N_SAMPLES = 10

# ---- Copie exacte du benchmark 0003 (cellule 3) : ne pas modifier, un test compare les deux ----
# --- normalisation TOLERANTE : minuscule + ponctuation ASCII retiree.
# On CONSERVE les caracteres baoule (ɛ ɔ ʼ, tons) : ils portent le sens. ---
_PUNCT = re.compile(r"[.,!?;:«»\"'`()\[\]{}…—–/\\]")
def normalize(s):
    s = unicodedata.normalize("NFC", (s or "")).lower().strip()
    s = _PUNCT.sub(" ", s)
    return re.sub(r"\s+", " ", s).strip()

def score(refs, hyps):
    R, H = [], []
    for r, h in zip(refs, hyps):
        rn = normalize(r)
        if rn:                       # jiwer plante sur reference vide
            R.append(rn); H.append(normalize(h))
    return {"cer": float(jiwer.cer(R, H)), "wer": float(jiwer.wer(R, H)), "n": len(R)}
# ---- fin de la copie ----


def find_checkpoint(root, step) -> Path:
    """Dossier `<step>/model` du checkpoint téléchargé (contient pp_00/tp_00/sdp_00.pt)."""
    hits = sorted({p.parent.parent.parent for p in Path(root).rglob("sdp_*.pt")
                   if p.parent.parent.parent.parent.name == step})
    if len(hits) != 1:
        raise SystemExit(f"{len(hits)} checkpoint(s) {step}/model trouvé(s) sous {root} : 1 attendu")
    return hits[0].resolve()


def write_card(asset_dir, step, model_dir) -> str:
    """Carte fairseq2 du modèle fine-tuné : hérite de BASE_CARD, remplace son checkpoint."""
    if not (Path(model_dir) / "pp_00").is_dir():
        raise SystemExit(f"{model_dir} n'est pas un checkpoint fairseq2 (pp_00/ absent)")
    name = f"wourri_bci_ctc_1b_{step}"
    asset_dir = Path(asset_dir)
    asset_dir.mkdir(parents=True, exist_ok=True)
    (asset_dir / f"{name}.yaml").write_text(
        f"name: {name}\nbase: {BASE_CARD}\ncheckpoint: {Path(model_dir).resolve().as_posix()}\n",
        encoding="utf-8",
    )
    return name


def load_test_clips(load_dataset=None, limit=None):
    """(phrases de référence, [(waveform, sample rate)]) du split test, dans l'ordre du dataset."""
    if load_dataset is None:
        from datasets import load_dataset
    rows = list(load_dataset(DATASET_REPO, revision=REV, split="test"))
    rows = rows[:limit] if limit else rows
    return [r["sentence"] for r in rows], [get_audio(r) for r in rows]


def _default_pipeline(card):
    from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline

    return ASRInferencePipeline(model_card=card)


def _free_gpu():
    gc.collect()
    try:
        import torch
    except ImportError:
        return
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def transcribe(card, paths, make_pipeline=None):
    """Transcriptions d'un modèle comme au benchmark 0003 (un clip à la fois, bci_Latn) + durées."""
    make_pipeline = make_pipeline or _default_pipeline
    t0 = time.time()
    pipe = make_pipeline(card)
    load_s = time.time() - t0
    t1 = time.time()
    hyps = list(pipe.transcribe(paths, lang=["bci_Latn"] * len(paths), batch_size=1))
    infer_s = time.time() - t1
    del pipe  # libère le modèle avant de charger le suivant (deux 1B ne tiennent pas sur un T4)
    _free_gpu()
    return hyps, round(load_s, 1), round(infer_s, 1)


def evaluate(models, out_path, load_dataset=None, make_pipeline=None, limit=None):
    """Mesure chaque (libellé, carte) sur les mêmes clips ; résultats écrits après chaque modèle."""
    import soundfile as sf

    refs, audios = load_test_clips(load_dataset, limit)
    tmp = tempfile.mkdtemp()
    paths = []
    for i, (wav, sr) in enumerate(audios):
        p = os.path.join(tmp, f"{i}.wav")
        sf.write(p, wav, sr)
        paths.append(p)
    out_path = Path(out_path)
    results = {}
    for label, card in models:
        hyps, load_s, infer_s = transcribe(card, paths, make_pipeline)
        r = score(refs, hyps)
        r.update(card=card, load_s=load_s, infer_s=infer_s,
                 empty_hyps=sum(1 for h in hyps if not normalize(h)),
                 samples=[{"ref": refs[i], "hyp": hyps[i]} for i in range(min(N_SAMPLES, len(hyps)))],
                 hyps=hyps)
        results[label] = r
        out_path.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"{label}: CER={r['cer'] * 100:.1f}%  WER={r['wer'] * 100:.1f}%  n={r['n']}  "
              f"vides={r['empty_hyps']}  (charg {load_s:.0f}s, infer {infer_s:.0f}s)", flush=True)
    return results


def control_ok(cer, n):
    """Le témoin (1B de départ) reproduit-il le benchmark 0003 : mêmes clips, ~22,1 % ?"""
    return n == BENCHMARK_0003_N and abs(cer - BENCHMARK_0003_CER) <= CONTROL_TOLERANCE


def report(results, base_label, ft_label):
    """Tableau, verdict du témoin, gain et exemples côte à côte (texte à recopier dans la fiche 0006)."""
    base, ft = results[base_label], results[ft_label]
    lines = [f"{'Modele':28s} {'CER':>7s} {'WER':>7s} {'n':>5s} {'vides':>6s}", "-" * 58]
    for label, r in results.items():
        lines.append(f"{label:28s} {r['cer'] * 100:6.1f}% {r['wer'] * 100:6.1f}% {r['n']:5d} {r['empty_hyps']:6d}")
    if control_ok(base["cer"], base["n"]):
        lines.append(f"\nTEMOIN OK : {base['cer'] * 100:.1f}% (benchmark 0003 : {BENCHMARK_0003_CER * 100:.1f}%)"
                     " -> comparaison recevable.")
    else:
        lines.append(f"\nTEMOIN ECART : {base['cer'] * 100:.1f}% sur {base['n']} clips, attendu "
                     f"{BENCHMARK_0003_CER * 100:.1f}% sur {BENCHMARK_0003_N} -> environnement different, "
                     "a signaler tel quel.")
    lines.append(f"Gain CER : {(base['cer'] - ft['cer']) * 100:+.1f} pts "
                 f"({base['cer'] * 100:.1f}% -> {ft['cer'] * 100:.1f}%)")
    lines.append("\n=== Exemples (reference | depart | fine-tune) ===")
    for i in range(min(len(base["samples"]), len(ft["samples"]))):
        lines.append(f"\n[{i}] REF : {base['samples'][i]['ref']}\n    DEP : {base['samples'][i]['hyp']}"
                     f"\n    FT  : {ft['samples'][i]['hyp']}")
    return "\n".join(lines)


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[0])
    ap.add_argument("--checkpoint-root", required=True, help="dossier où le checkpoint a été téléchargé")
    ap.add_argument("--step", default="step_1200", help="étape retenue sur dev (fiche 0006)")
    ap.add_argument("--asset-dir", default=None, help="répertoire de cartes fairseq2 (défaut : temporaire)")
    ap.add_argument("--out", default="eval_bci_cer_results.json")
    ap.add_argument("--limit", type=int, default=None, help="essai rapide sur N clips (défaut : les 290)")
    args = ap.parse_args(argv)

    asset_dir = args.asset_dir or tempfile.mkdtemp(prefix="fairseq2_assets_")
    card = write_card(asset_dir, args.step, find_checkpoint(args.checkpoint_root, args.step))
    # fairseq2 lit ce répertoire à sa première utilisation : à poser avant tout chargement de modèle.
    os.environ["FAIRSEQ2_USER_ASSET_DIR"] = str(asset_dir)

    base_label, ft_label = "temoin 1B depart", f"1B fine-tune {args.step}"
    results = evaluate([(base_label, BASE_CARD), (ft_label, card)], args.out, limit=args.limit)
    print("\n" + report(results, base_label, ft_label))
    print("\nResultats complets ->", args.out)


if __name__ == "__main__":
    main()


## Étape 3 — Télécharger le modèle fine-tuné (dépôt privé, ~3,9 Go)

Sur Kaggle, le jeton est lu dans les **Secrets** (`HF_TOKEN`). Ailleurs (RunPod), une **case de saisie masquée**
apparaît sous la cellule : colle le jeton dedans, puis Entrée. Il sert à ce seul téléchargement, puis il est oublié.

In [ ]:
# Cellule 6 — télécharger le checkpoint retenu depuis Hugging Face (jeton lu dans les Secrets)
from huggingface_hub import snapshot_download

try:
    from kaggle_secrets import UserSecretsClient
    _token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:                          # hors Kaggle, ou secret absent : saisie masquée
    import getpass
    _token = getpass.getpass("Jeton Hugging Face (lecture seule) : ")
try:
    snapshot_download(HF_REPO, allow_patterns=[f"{STEP}/*"], local_dir=str(CKPT_ROOT), token=_token)
finally:
    del _token
shards = sorted(CKPT_ROOT.rglob("sdp_*.pt"))
for s in shards:
    print(s.relative_to(CKPT_ROOT), f"{s.stat().st_size / 2**30:.2f} Gio")
if not shards:
    raise SystemExit("Aucun fichier sdp_*.pt téléchargé : vérifier HF_REPO, STEP et le secret HF_TOKEN.")
print(f"\n>>> MODÈLE {STEP} TÉLÉCHARGÉ -> cellule 7.")

## Étape 4 — La mesure (~10 à 15 min)

Deux modèles, l'un après l'autre, sur les **290 clips** : le **témoin** (1B de départ) puis le **fine-tuné**.
Chacun affiche une ligne `CER=… WER=…` à la fin de sa passe ; le tableau final, le verdict du témoin, le gain et
10 exemples côte à côte s'affichent ensuite. **C'est fini quand `>>> MESURE TERMINÉE` apparaît.**

In [ ]:
# Cellule 7 — mesure : témoin (1B de départ) puis 1B fine-tuné, mêmes 290 clips, protocole du benchmark 0003
# Un seul GPU ; PCI_BUS_ID : CUDA numérote les GPU comme nvidia-smi.
env = clean_env({"CUDA_DEVICE_ORDER": "PCI_BUS_ID", "CUDA_VISIBLE_DEVICES": "0"})
sh([VENV_PY, "eval_bci_cer.py", "--checkpoint-root", CKPT_ROOT, "--step", STEP, "--out", RESULTS],
   env=env, log_path=WORK / "eval_bci_cer.log")
print("\n>>> MESURE TERMINÉE. Copie tout le texte affiché depuis « Modele   CER » jusqu'ici et envoie-le.")

## Fin

1. **Copie** le texte affiché par la cellule 7 (tableau, verdict du témoin, gain, exemples) et envoie-le : il sera
   reporté dans `docs/benchmarks/0006-omnilingual-1b-finetune-resultats.md`.
2. Les résultats complets (les 290 transcriptions de chaque modèle) sont dans `eval_bci_cer_results.json`
   (Kaggle : panneau *Output* à droite ; RunPod : panneau de fichiers de Jupyter, clic droit → *Download*).
3. Kaggle : *Run → Stop session*. RunPod : **Stop** puis **Terminate** le pod (sinon il reste facturé).